# Part 3: pi0-FAST — The Tokenization Experiment

## Notebook 8 — The FAST Pipeline Step by Step

FAST (Frequency-space Action Sequence Tokenization) converts continuous action chunks into discrete tokens via:

1. **Normalize** action chunk to [-1, 1]
2. **DCT** (Discrete Cosine Transform) → frequency domain
3. **Quantize** coefficients to integers
4. **Flatten** into 1D, low-frequency first
5. **BPE** compress into dense tokens

We load the pre-trained FAST tokenizer and walk through each step.


### 1. Load the FAST tokenizer


In [ ]:
from transformers import AutoProcessor

# Load the universal FAST+ tokenizer trained on 1M action sequences
tokenizer = AutoProcessor.from_pretrained(
    "physical-intelligence/fast", trust_remote_code=True
)
print(f"FAST tokenizer loaded")


### 2. Create a sample action chunk


In [ ]:
import numpy as np

time_horizon = 50  # 1 second at 50 Hz
action_dim = 7    # x, y, z, roll, pitch, yaw, gripper

# Sample action: sinusoidal joint movement
t = np.linspace(0, 1, time_horizon)
actions = np.zeros((time_horizon, action_dim))
actions[:, 0] = 0.5 * np.sin(2 * np.pi * 1.5 * t)   # x: sinusoidal
actions[:, 1] = 0.3 * np.cos(2 * np.pi * 2.0 * t)   # y: cosine
actions[:, 2] = 0.1 * t                              # z: linear
actions[:, 3] = 0.2 * np.sin(2 * np.pi * 0.5 * t)   # roll
actions[:, 4] = -0.1 * t                             # pitch: linear
actions[:, 5] = 0.05 * np.sin(2 * np.pi * 3.0 * t)  # yaw: fast
actions[:, 6] = np.where(t > 0.5, 1.0, 0.0)         # gripper: step

print(f"Action chunk shape: {actions.shape}")
print(f"Value range: [{actions.min():.3f}, {actions.max():.3f}]")


### 3. Visualize the action chunk


In [ ]:
import matplotlib.pyplot as plt

dim_names = ["x", "y", "z", "roll", "pitch", "yaw", "gripper"]
fig, axes = plt.subplots(4, 2, figsize=(12, 8))
axes = axes.flatten()
for i in range(action_dim):
    axes[i].plot(t, actions[:, i])
    axes[i].set_title(f'{dim_names[i]}')
    axes[i].set_xlabel('Time (s)')
    axes[i].grid(True, alpha=0.3)
axes[-1].axis('off')
fig.suptitle('Sample Action Chunk (50 timesteps × 7 dims)', fontsize=14)
plt.tight_layout()
plt.show()


### 4. Tokenize with FAST

The FAST tokenizer compresses this 50×7 = 350-value chunk into a small number of discrete tokens.


In [ ]:
fast_tokens = tokenizer(padding=False)[:20]
print(f"Action values: {actions.shape} = {actions.size} floats")
print(f"FAST tokens:   {len(fast_tokens)} tokens")
print(f"Compression:   {actions.size / len(fast_tokens):.1f}×")

# Show first few tokens
print(f"\nFirst 10 tokens: {fast_tokens[:10]}")


### 5. DCT step: time domain → frequency domain

The Discrete Cosine Transform (same as JPEG) converts the action signal to frequency domain. Smooth motions concentrate energy in low-frequency coefficients — most high-frequency coefficients are near zero (sparse).


In [ ]:
from scipy.fftpack import dct

# Apply DCT along time axis (axis=0)
dct_coeffs = dct(actions, axis=0, norm='ortho')
print(f"DCT coefficient shape: {dct_coeffs.shape}")
print(f"Non-zero coeffs: {np.count_nonzero(np.abs(dct_coeffs) > 1e-6)}")

# Energy concentration: how much energy in first K coefficients?
energy = np.sum(dct_coeffs ** 2, axis=0)
for k in [5, 10, 20]:
    pct = 100 * np.sum(dct_coeffs[:k] ** 2, axis=0) / energy
    print(f"  First {k:2d} coeffs capture: {pct.mean():.1f}% of energy")


### 6. Quantize: float → int


In [ ]:
# Quantization step (scaled and rounded)
scale = 1000  # FAST uses scaling to preserve precision
quantized = np.round(dct_coeffs * scale).astype(np.int32)
print(f"Quantized DCT shape: {quantized.shape}")
print(f"Value range: [{quantized.min()}, {quantized.max()}]")


### 7. BPE: compress sparse coefficients into dense tokens


In [ ]:
# After flattening: 50*7=350 coefficients → BPE merges → ~30-60 tokens
# The BPE tokenizer was trained on DCT coefficients from 1M action sequences
# It learns which coefficient patterns commonly appear together

print("BPE compression:")
print("  Input:  350 quantized DCT coefficients")
print("  Output: ~30-60 BPE tokens")
print("  Ratio:  ~6-12× compression")

# Compare: if we used RT-1 style binning
tokens_binned = time_horizon * action_dim  # 350
tokens_fast = len(fast_tokens)  # ~30-60
print(f"\nRT-1/RT-2 binning: {tokens_binned} tokens")
print(f"FAST:             {tokens_fast} tokens")
print(f"Compression:      {tokens_binned / tokens_fast:.1f}×")


### 8. Round-trip: decode back to actions


In [ ]:
# FAST is fully invertible
decoded = tokenizer.decode(fast_tokens)
# decoded should approximately match the original actions
print(f"Decoded shape: {decoded.shape}")

# Reconstruction error
if decoded.shape == actions.shape:
    mse = np.mean((decoded - actions) ** 2)
    print(f"Reconstruction MSE: {mse:.6f}")
else:
    print(f"Shape mismatch — tokenizer may have different horizon/dim")


### To Summarize

FAST compresses action chunks via DCT + BPE, achieving 10× fewer tokens than naive per-dimension binning. The compression works because smooth motions are sparse in the frequency domain. In the next notebook, we'll see how pi0-FAST uses these tokens for autoregressive training.
